<a href="https://colab.research.google.com/github/racoope70/daytrading-with-ml/blob/main/implement_market_wizard_strategies_v3_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade --force-reinstall \
    dask==2024.11.2 \
    rapids-dask-dependency==24.12.0 \
    cudf-cu12==24.12.0 \
    cuml-cu12==24.12.0 \
    pylibraft-cu12==24.12.0 \
    pylibcudf-cu12==24.12.0 \
    numba==0.61.0 \
    stable-baselines3[extra] \
    gymnasium==0.29.1 \
    gym-anytrading==2.0.0

In [1]:
!pip install gymnasium[box2d] stable-baselines3[extra] gym-anytrading

In [2]:
!pip install yfinance xgboost joblib protobuf==3.20.3

In [3]:
!pip install --upgrade protobuf
!pip install protobuf==3.20.3
!pip install tensorflow

In [4]:
pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu121


In [5]:
pip install --force-reinstall cudf-cu12 cuml-cu12 rapids-dask-dependency

In [1]:
#Core Libraries
import os
import time
import gc
import numpy as np
import pandas as pd
import scipy
import numba
import matplotlib.pyplot as plt
from collections import defaultdict

#Machine Learning & Data Processing
import xgboost as xgb
import yfinance as yf
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE

#RAPIDS Libraries (cuDF & cuML for GPU acceleration)
import cudf
import cuml
import dask

#Reinforcement Learning (Stable Baselines3)
import torch
import stable_baselines3
from stable_baselines3 import A2C, PPO, DDPG, TD3
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.env_util import make_vec_env

#Gym & Trading Environments
import gymnasium as gym  #Use Gymnasium instead of Gym
from gymnasium.spaces import Discrete, Box
from gymnasium.wrappers import TimeLimit
import gym_anytrading
from gym_anytrading.envs import StocksEnv

#TensorFlow & GPU Optimization
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

#Set CUDA Paths
os.environ['CUDA_HOME'] = '/usr/local/cuda-11.8'
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-11.8/lib64'

#Print Version Check
print("cuDF Version:", cudf.__version__)
print("cuML Version:", cuml.__version__)
print("Dask Version:", dask.__version__)
print("Stable Baselines3 Installed:", stable_baselines3.__version__)
print("Gymnasium Version:", gym.__version__)
print("NumPy Version:", np.__version__)
print("SciPy Version:", scipy.__version__)
print("Pandas Version:", pd.__version__)

#GPU Check
!nvidia-smi


In [2]:
#Define Discrete Trading Environment
class DiscreteTradingEnv(gym.Env):
    def __init__(self, df, frame_bound=(10, 100), window_size=10, verbose=False):
        super(DiscreteTradingEnv, self).__init__()
        self.df = df
        self.frame_bound = frame_bound
        self.window_size = window_size
        self.current_step = self.frame_bound[0]
        self.done = False
        self.verbose = verbose

        #Portfolio & Trading Variables
        self.initial_balance = 100000
        self.portfolio_value = self.initial_balance
        self.shares_held = 0
        self.last_trade_price = 0
        self.position_size = 0.1  # 10% of the portfolio per trade

        #Logging Trades & Rewards
        self.trade_log = []
        self.rewards_log = []

        #Define Action and Observation Space
        self.action_space = Discrete(3)  # Actions: 0 = SELL, 1 = HOLD, 2 = BUY
        self.observation_space = Box(low=-np.inf, high=np.inf, shape=(window_size + 2,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = self.frame_bound[0]
        self.done = False

        #Reset Portfolio
        self.portfolio_value = self.initial_balance
        self.shares_held = 0
        self.last_trade_price = 0

        #Clear Logs
        self.trade_log.clear()
        self.rewards_log.clear()

        return self._next_observation(), {}

    def step(self, action):
        if self.current_step >= len(self.df) - 1:
            self.done = True
            return self._next_observation(), 0, self.done, False, {}

        self.current_step += 1
        new_price = self.df['Close'].iloc[self.current_step]

        reward = 0  # Default reward
        executed = False  # Track if a trade was executed

        #BUY ACTION (Require a Bigger Price Drop Before Buying)
        if action == 2:
            allocated_funds = min(self.portfolio_value * self.position_size, self.portfolio_value * 0.3)
            if self.portfolio_value >= allocated_funds:
                shares_bought = allocated_funds / new_price
                self.shares_held += shares_bought
                self.portfolio_value -= shares_bought * new_price
                self.last_trade_price = new_price
                executed = True

                #Adjusted Reward Logic for Better Training
                price_change = (self.df['Close'].iloc[self.current_step - 1] - new_price) / max(new_price, 1e-6)
                if price_change > 0.01:  # 1%+ Drop → High Reward
                    reward = price_change * 80
                elif price_change > 0.005:  # 0.5%+ Drop → Moderate Reward
                    reward = price_change * 60
                else:
                    reward = 0.003  #No negative BUY rewards!

        #SELL ACTION (Encourage Profitable Selling)
        elif action == 0 and self.shares_held > 0:
            sell_value = self.shares_held * new_price
            profit = (new_price - self.last_trade_price) * self.shares_held

            #Require at Least 2% Profit Before Selling
            profit_percent = (new_price - self.last_trade_price) / max(self.last_trade_price, 1e-6)
            if profit_percent > 0.02:
                reward = profit_percent * 350  # Strong reward for good sales
            else:
                reward = profit_percent * 10  # Reduce penalty for small losses

            #Reset portfolio after calculation
            self.portfolio_value += sell_value
            self.shares_held = 0
            self.last_trade_price = 0
            executed = True

        #HOLD ACTION (Encourage Holding If Profitable)
        else:
            unrealized_profit = (new_price - self.last_trade_price) * self.shares_held
            reward = np.tanh(unrealized_profit / self.initial_balance) * 5

        #Log Trade
        self.trade_log.append({
            "Step": self.current_step,
            "Action": ["SELL", "HOLD", "BUY"][action],
            "Shares Held": self.shares_held,
            "Portfolio Value": self.portfolio_value,
            "Stock Price": new_price,
            "Reward": reward
        })

        self.rewards_log.append(reward)

        return self._next_observation(), reward, self.done, False, {}

    def _next_observation(self):
        stock_prices = np.array(self.df['Close'].iloc[self.current_step - self.window_size:self.current_step], dtype=np.float32)
        return np.concatenate(([self.portfolio_value], stock_prices, [self.shares_held]))


In [3]:
#Ensure TensorFlow GPU Memory Allocation is Configured
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  # Prevents full allocation
        print("TensorFlow GPU memory growth enabled")
    except RuntimeError as e:
        print(f"TensorFlow GPU memory issue: {e}")

#CUDA Paths
os.environ['CUDA_HOME'] = '/usr/local/cuda-11.8'
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-11.8/lib64'



def download_stock_data(ticker, period="720d", interval="1h", max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Attempt {attempt}: Downloading {ticker} stock data...")
            df_live = yf.download(ticker, period=period, interval=interval)
            if not df_live.empty:
                print("Successfully downloaded stock data!")
                df_live.reset_index(inplace=True)
                return df_live
            raise ValueError("Downloaded data is empty. Retrying...")
        except Exception as e:
            print(f"Error: {e}. Retrying in {attempt * 5} seconds...")
            time.sleep(attempt * 5)
    print("Failed to download stock data after multiple attempts.")
    return None

df_live = download_stock_data("AAPL")
if df_live is None:
    print("Using previously saved dataset instead.")
    file_path = '/content/drive/My Drive/aaplfeature_engineered_dataset.csv'
    df_live = pd.read_csv(file_path)

df = df_live.copy()


TensorFlow GPU memory growth enabled
YF.download() has changed argument auto_adjust default to True


Successfully downloaded stock data!


In [4]:
#Fix Missing Index
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

In [5]:
#Step 3: Feature Engineering
def compute_technical_indicators(df):
    #Simple Moving Average (SMA) & Bollinger Bands
    df['SMA_20'] = df['Close'].rolling(window=20).mean()
    df['STD_20'] = df['Close'].rolling(window=20).std()

    df['Upper_Band'] = df['SMA_20'] + 2 * df['STD_20']
    df['Lower_Band'] = df['SMA_20'] - 2 * df['STD_20']  #Added Lower Band

    #Stochastic Oscillator
    df['Lowest_Low'] = df['Low'].rolling(window=14).min()
    df['Highest_High'] = df['High'].rolling(window=14).max()
    df['Stoch'] = ((df['Close'] - df['Lowest_Low']) / (df['Highest_High'] - df['Lowest_Low'])) * 100

    #Rolling Volatility Feature
    df['volatility'] = df['Close'].pct_change().rolling(20).std()

    #Drop NA values after feature calculations
    df.dropna(inplace=True)

    return df

#Step 4: Labeling (Buy/Sell Signals)
def generate_trade_labels(df, lookahead=10, threshold_factor=2):
    #Ensure 'Close' column exists
    if 'Close' not in df.columns:
        raise KeyError("'Close' column is missing. Cannot generate trade labels.")

    #Generate future price shift
    df['Future_Close'] = df['Close'].shift(-lookahead)
    df['Price_Change'] = (df['Future_Close'] - df['Close']) / df['Close']

    #Primary Target Label: Binary Classification (Buy = 1, Sell = 0)
    df['Target'] = np.where(df['Price_Change'] > 0.03, 1, 0)

    #Volatility-Adjusted Dynamic Labels
    buy_threshold = df['volatility'] * threshold_factor
    sell_threshold = -df['volatility'] * threshold_factor

    df['Dynamic_Label'] = np.where(df['Price_Change'] > buy_threshold, 1,
                            np.where(df['Price_Change'] < sell_threshold, -1, 0))

    #Drop NaN values after target calculations
    df.dropna(inplace=True)
    return df

#Apply Feature Engineering & Target Labeling
df = compute_technical_indicators(df)  #Compute Features
df = generate_trade_labels(df)  #Generate 'Target' Column

#Check if 'Target' exists before training
if 'Target' not in df.columns:
    raise KeyError("'Target' column is missing after feature engineering. Check generate_trade_labels(df).")

In [6]:
#Ensure df is preprocessed correctly
if "SMA_20" not in df.columns:
    raise ValueError("'SMA_20' feature missing. Ensure feature engineering is applied.")

#Define Features for Model Training
features = ['SMA_20', 'STD_20', 'Upper_Band', 'Lower_Band', 'Stoch', 'volatility']
target_column = 'Target'


In [7]:
def train_walk_forward(df, features, label='Target', model_path="rf_trading_model.pkl"):
    """Train a Random Forest model using time-based walk-forward validation."""

    #Ensure train-test splits are time-based
    tscv = TimeSeriesSplit(n_splits=5)
    accuracy_scores = []

    for train_index, test_index in tscv.split(df):
        train, test = df.iloc[train_index], df.iloc[test_index]

        #Convert pandas to cudf for GPU acceleration (ensure float32)
        X_train = cudf.DataFrame.from_pandas(train[features]).astype(np.float32)
        y_train = cudf.Series(train[label].astype(np.float32))

        X_test = cudf.DataFrame.from_pandas(test[features]).astype(np.float32)
        y_test = cudf.Series(test[label].astype(np.float32))

        #Convert to Pandas before using SMOTE
        smote = SMOTE(sampling_strategy='auto', random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X_train.to_pandas(), y_train.to_pandas())

        #Convert back to cuDF
        X_train = cudf.DataFrame.from_pandas(X_resampled)
        y_train = cudf.Series(y_resampled)

        #Ensure data is available before training
        if X_train.shape[0] == 0:
            raise ValueError("No training data available! Ensure features are correctly calculated.")

        #Train model
        model = RandomForestClassifier(n_estimators=100)
        model.fit(X_train.to_pandas(), y_train.to_pandas())  #Explicit conversion

        #Save the model
        joblib.dump(model, model_path)  #Save the model to a file

        #Convert X_test to Pandas before prediction
        probs = model.predict_proba(X_test.to_pandas())  #Fix: Convert before calling predict_proba
        custom_threshold = 0.4  # Adjust if needed
        preds = (probs[:, 1] > custom_threshold).astype(int)

        #Convert y_test to Pandas before using NumPy functions
        y_test = y_test.to_pandas().to_numpy()  #Fix: Convert cuDF Series → Pandas → NumPy

        acc = accuracy_score(y_test, preds)
        accuracy_scores.append(acc)

    print(f"Avg Accuracy Across Time Splits: {np.mean(accuracy_scores):.4f}")

    #Print Classification Report
    print("\nRandom Forest Classification Report:")
    print(classification_report(y_test, preds))

    print(f"Random Forest model saved as {model_path}")
    return model

#Train & Save Random Forest Model
rf_model = train_walk_forward(df, features, label='Target')


Avg Accuracy Across Time Splits: 0.9048

Random Forest Classification Report:
              precision    recall  f1-score   support

         0.0       0.97      0.98      0.98       808
         1.0       0.00      0.00      0.00        22

    accuracy                           0.95       830
   macro avg       0.49      0.49      0.49       830
weighted avg       0.95      0.95      0.95       830

Random Forest model saved as rf_trading_model.pkl


In [8]:
#Ensure Feature Engineering is Applied Before Training
if "SMA_20" not in df.columns:  # Prevent recomputation
    df = compute_technical_indicators(df)

df = generate_trade_labels(df)  #Generate Buy/Sell Labels

#Ensure Features & Trade Signals Exist
required_features = ['SMA_20', 'STD_20', 'Upper_Band', 'Lower_Band', 'Stoch', 'volatility']
if not all(feature in df.columns for feature in required_features):
    raise ValueError(f"Missing Features: {set(required_features) - set(df.columns)}. Run feature engineering first!")

if 'Target' not in df.columns:
    raise ValueError("Target column is missing! Run generate_trade_labels(df) first!")

print("Features and labels are ready for XGBoost training.")

#Drop NaN Values Before Training
df.dropna(subset=required_features + ['Target'], inplace=True)

#Define Feature Columns & Target
feature_columns = required_features
target_column = 'Target'

Features and labels are ready for XGBoost training.


In [9]:
def train_xgboost(df, features, label='Target', model_path="xgb_trading_model.pkl"):
    tscv = TimeSeriesSplit(n_splits=5)
    accuracy_scores = []

    for train_index, test_index in tscv.split(df):
        train, test = df.iloc[train_index], df.iloc[test_index]

        #Keep data on GPU
        X_train = cudf.DataFrame(train[features]).astype(np.float32)
        y_train = cudf.Series(train[label].astype(np.float32))

        X_test = cudf.DataFrame(test[features]).astype(np.float32)
        y_test = cudf.Series(test[label].astype(np.float32))

        #Fix scale_pos_weight calculation
        scale_pos_weight = (sum(y_train.to_numpy() == 0) / (sum(y_train.to_numpy() == 1) + 1e-6))

        #Use latest GPU-optimized settings
        params = {
            'objective': 'binary:logistic',
            'learning_rate': 0.1,
            'n_estimators': 50,
            'tree_method': 'hist',  #Use "hist" instead of "gpu_hist"
            'device': 'cuda',       #Explicitly set GPU device
            'scale_pos_weight': scale_pos_weight,
            'random_state': 42
        }

        #Convert cuDF to NumPy for XGBoost compatibility
        X_train_np = X_train.to_numpy()
        y_train_np = y_train.to_numpy()

        X_test_np = X_test.to_numpy()
        y_test_np = y_test.to_numpy()

        #Train model
        model = xgb.XGBClassifier(**params)
        model.fit(X_train_np, y_train_np)

        #Save the model
        joblib.dump(model, model_path)  #Save the model to a file

        #Use probability threshold for better predictions
        probs = model.predict_proba(X_test_np)
        custom_threshold = 0.4  #Adjust if needed
        preds = (probs[:, 1] > custom_threshold).astype(int)

        acc = accuracy_score(y_test_np, preds)
        accuracy_scores.append(acc)

    print(f"Avg Accuracy Across Time Splits: {np.mean(accuracy_scores):.4f}")

    #Print Classification Report
    print("\nXGBoost Classification Report:")
    print(classification_report(y_test_np, preds))

    print(f"XGBoost model saved as {model_path}")
    return model

#Train & Save XGBoost Model
xgb_model = train_xgboost(df, feature_columns, label=target_column)

#Free Memory
gc.collect()

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [14:23:19] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


Avg Accuracy Across Time Splits: 0.9123

XGBoost Classification Report:
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98       805
         1.0       0.08      0.04      0.06        23

    accuracy                           0.96       828
   macro avg       0.53      0.51      0.52       828
weighted avg       0.95      0.96      0.95       828

XGBoost model saved as xgb_trading_model.pkl


239

In [10]:
#Define Discrete Trading Environment
class DiscreteTradingEnv(gym.Env):
    def __init__(self, df, frame_bound=(10, 100), window_size=10, verbose=False):
        super(DiscreteTradingEnv, self).__init__()
        self.df = df
        self.frame_bound = frame_bound
        self.window_size = window_size
        self.current_step = self.frame_bound[0]
        self.done = False
        self.verbose = verbose

        #Portfolio & Trading Variables
        self.initial_balance = 100000
        self.portfolio_value = self.initial_balance
        self.shares_held = 0
        self.last_trade_price = 0
        self.position_size = 0.1  # 10% of the portfolio per trade

        #Logging Trades & Rewards
        self.trade_log = []
        self.rewards_log = []

        #Define Action and Observation Space
        self.action_space = Discrete(3)  # Actions: 0 = SELL, 1 = HOLD, 2 = BUY
        self.observation_space = Box(low=-np.inf, high=np.inf, shape=(window_size + 2,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = self.frame_bound[0]
        self.done = False

        #Reset Portfolio
        self.portfolio_value = self.initial_balance
        self.shares_held = 0
        self.last_trade_price = 0

        #Clear Logs
        self.trade_log.clear()
        self.rewards_log.clear()

        return self._next_observation(), {}

    def step(self, action):
        if self.current_step >= len(self.df) - 1:
            self.done = True
            return self._next_observation(), 0, self.done, False, {}

        self.current_step += 1
        new_price = self.df['Close'].iloc[self.current_step]

        reward = 0  # Default reward
        executed = False  # Track if a trade was executed

        #BUY ACTION (Require a Bigger Price Drop Before Buying)
        if action == 2:
            allocated_funds = min(self.portfolio_value * self.position_size, self.portfolio_value * 0.3)
            if self.portfolio_value >= allocated_funds:
                shares_bought = allocated_funds / new_price
                self.shares_held += shares_bought
                self.portfolio_value -= shares_bought * new_price
                self.last_trade_price = new_price
                executed = True

                #Adjusted Reward Logic for Better Training
                price_change = (self.df['Close'].iloc[self.current_step - 1] - new_price) / max(new_price, 1e-6)
                if price_change > 0.01:  # 1%+ Drop → High Reward
                    reward = price_change * 80
                elif price_change > 0.005:  # 0.5%+ Drop → Moderate Reward
                    reward = price_change * 60
                else:
                    reward = 0.003  #No negative BUY rewards!

        #SELL ACTION (Encourage Profitable Selling)
        elif action == 0 and self.shares_held > 0:
            sell_value = self.shares_held * new_price
            profit = (new_price - self.last_trade_price) * self.shares_held

            #Require at Least 2% Profit Before Selling
            profit_percent = (new_price - self.last_trade_price) / max(self.last_trade_price, 1e-6)
            if profit_percent > 0.02:
                reward = profit_percent * 350  # Strong reward for good sales
            else:
                reward = profit_percent * 10  # Reduce penalty for small losses

            #Reset portfolio after calculation
            self.portfolio_value += sell_value
            self.shares_held = 0
            self.last_trade_price = 0
            executed = True

        #HOLD ACTION (Encourage Holding If Profitable)
        else:
            unrealized_profit = (new_price - self.last_trade_price) * self.shares_held
            reward = np.tanh(unrealized_profit / self.initial_balance) * 5

        #Log Trade
        self.trade_log.append({
            "Step": self.current_step,
            "Action": ["SELL", "HOLD", "BUY"][action],
            "Shares Held": self.shares_held,
            "Portfolio Value": self.portfolio_value,
            "Stock Price": new_price,
            "Reward": reward
        })

        self.rewards_log.append(reward)

        return self._next_observation(), reward, self.done, False, {}

    def _next_observation(self):
        stock_prices = np.array(self.df['Close'].iloc[self.current_step - self.window_size:self.current_step], dtype=np.float32)
        return np.concatenate(([self.portfolio_value], stock_prices, [self.shares_held]))

#Step 1: Define Environment Creation Function
def make_env():
    return gym.wrappers.TimeLimit(
        DiscreteTradingEnv(df=df, frame_bound=(10, len(df)), window_size=10),
        max_episode_steps=1000
    )

#Step 2: Properly Create Vectorized Environment
env_discrete = make_vec_env(make_env, n_envs=1)

#Step 3: Normalize Environment (Fixed Method)
env_discrete = VecNormalize(env_discrete, norm_obs=True, norm_reward=True, clip_obs=10.0)

#Step 4: Debugging Action Space
print(f"Action Space: {env_discrete.action_space}")
print(f"Sample Action: {env_discrete.action_space.sample()}")

#Step 5: Train PPO Model
print("\nTraining PPO Model...")
ppo_model = PPO("MlpPolicy", env_discrete, verbose=1, device="cuda")
ppo_model.learn(total_timesteps=10000)
ppo_model.save("ppo_trading_model_v1")
print("\nPPO Training Complete!")



PPO Training Complete!


In [11]:
#Retrain A2C Model with Higher Entropy Regularization
print("\nRetraining A2C Model with Higher Entropy Regularization...")
a2c_model = A2C(
    "MlpPolicy",
    env_discrete,
    learning_rate=0.0005,  #Lower learning rate for more stable updates
    gamma=0.98,  #Encourage slightly longer-term rewards
    vf_coef=0.4,  #Balance value function loss
    ent_coef=0.01,  #Increase entropy for better exploration
    verbose=1,
    device="cuda"
)

print("\nRetraining A2C Model with Adjusted Parameters...")
a2c_model.learn(total_timesteps=10000)  #Train for 100000 timesteps
a2c_model.save("a2c_trading_model_v1")
print("\nA2C Training Complete!")


A2C Training Complete!


In [14]:
#Define Model Evaluation Function (Fixed for VecEnv)
def evaluate_model(model, env, num_steps=100):
    """Evaluates a trained model and logs actions taken."""
    obs = env.reset()  #FIX: No unpacking for VecEnv reset
    action_history = []
    total_rewards = np.zeros(env.num_envs)  #VecEnv handles multiple environments

    for step in range(num_steps):
        action, _ = model.predict(obs)
        action = int(action.item())  #Convert to integer (Discrete Action)

        print(f"Executing action: {action}")  # Debugging

        obs, reward, done, _ = env.step([action])  #FIX: Wrap action in list for VecEnv
        action_history.append(action)
        total_rewards += reward  #Correctly sum rewards across all environments

        if step % 10 == 0:
            print(f"Step {step}: Action: {action}, Reward: {reward.mean():.4f}")

        if done.any():  #FIX: `done` is an array in VecEnv
            obs = env.reset()

    #Print action history and total test reward
    unique_actions = set(action_history)
    print(f"Actions Taken: {action_history if action_history else 'No trades executed.'}")
    print(f"Total Test Reward: {total_rewards.sum():.4f}")  #FIX: Sum over VecEnv rewards

    if len(unique_actions) < 3:
        print("Agent might not be exploring all actions properly.")

#Evaluate PPO Model
print("\nEvaluating PPO Model...")
evaluate_model(ppo_model, env_discrete)

#Evaluate A2C Model
print("\nEvaluating A2C Model...")
evaluate_model(a2c_model, env_discrete)

print("\nPPO & A2C Evaluation Complete!")

#Get the first (and only) wrapped discrete environment
wrapped_env = env_discrete.envs[0]

#Properly Access Trade Log (Backward Compatible)
if hasattr(env_discrete, "get_wrapper_attr"):
    trade_log_ppo = env_discrete.get_wrapper_attr("trade_log")
    rewards_log_ppo = env_discrete.get_wrapper_attr("rewards_log")

    trade_log_a2c = env_discrete.get_wrapper_attr("trade_log")
    rewards_log_a2c = env_discrete.get_wrapper_attr("rewards_log")
else:
    trade_log_ppo = wrapped_env.unwrapped.trade_log if hasattr(wrapped_env, "trade_log") else []
    rewards_log_ppo = wrapped_env.unwrapped.rewards_log if hasattr(wrapped_env, "rewards_log") else []

    trade_log_a2c = wrapped_env.unwrapped.trade_log if hasattr(wrapped_env, "trade_log") else []
    rewards_log_a2c = wrapped_env.unwrapped.rewards_log if hasattr(wrapped_env, "rewards_log") else []

#Summarize Trade Results
num_trades_ppo = len(trade_log_ppo)
num_rewards_ppo = len(rewards_log_ppo)
num_trades_a2c = len(trade_log_a2c)
num_rewards_a2c = len(rewards_log_a2c)

print(f"Number of Trades Logged (PPO): {num_trades_ppo}")
print(f"Number of Rewards Logged (PPO): {num_rewards_ppo}")

#Log Mean & Max Rewards (More Insight)
if rewards_log_ppo:
    print(f"PPO Mean Reward: {np.mean(rewards_log_ppo):.4f}, Max Reward: {np.max(rewards_log_ppo):.4f}")

if num_trades_ppo == 0:
    print("No trades recorded. The PPO agent might not be executing actions.")
if num_rewards_ppo == 0:
    print("No rewards recorded. The PPO environment might not be returning meaningful rewards.")

print(f"Number of Trades Logged (A2C): {num_trades_a2c}")
print(f"Number of Rewards Logged (A2C): {num_rewards_a2c}")

if rewards_log_a2c:
    print(f"A2C Mean Reward: {np.mean(rewards_log_a2c):.4f}, Max Reward: {np.max(rewards_log_a2c):.4f}")

if num_trades_a2c == 0:
    print("No trades recorded. The A2C agent might not be executing actions.")
if num_rewards_a2c == 0:
    print("No rewards recorded. The A2C environment might not be returning meaningful rewards.")

#Detect Stagnant Portfolio Value
if num_trades_ppo > 0:
    portfolio_changes = [t["Portfolio Value"] for t in trade_log_ppo]
    if len(set(portfolio_changes)) == 1:
        print("PPO agent's portfolio value is not changing. Possible issue with trading logic.")



Evaluating PPO Model...
Step 0: Action: 2, Reward: 0.0005
Step 10: Action: 2, Reward: 0.0005
Step 20: Action: 1, Reward: -0.0001
Step 30: Action: 0, Reward: -0.0129
Step 40: Action: 0, Reward: -0.0627
Step 50: Action: 0, Reward: -0.0243
Step 60: Action: 0, Reward: 0.0000
Step 70: Action: 1, Reward: 0.0000
Step 80: Action: 2, Reward: 0.0005
Step 90: Action: 1, Reward: -0.0001
Total Test Reward: 3.8596

Evaluating A2C Model...
Step 0: Action: 2, Reward: 0.0005
Step 10: Action: 2, Reward: 0.0005
Step 20: Action: 2, Reward: 0.0005
Step 30: Action: 2, Reward: 0.0782
Step 40: Action: 2, Reward: 0.1929
Step 50: Action: 2, Reward: 0.0005
Step 60: Action: 2, Reward: 0.0005
Step 70: Action: 2, Reward: 0.0543
Step 80: Action: 2, Reward: 0.0005
Step 90: Action: 2, Reward: 0.0005
Total Test Reward: 2.4206

PPO & A2C Evaluation Complete!
Number of Trades Logged (PPO): 100
Number of Rewards Logged (PPO): 100
PPO Mean Reward: 0.1498, Max Reward: 1.8002
Number of Trades Logged (A2C): 100
Number of Rew

In [15]:
gc.collect()
torch.cuda.empty_cache()

In [16]:
class SARSAAgent:
    def __init__(self, env, alpha=0.1, gamma=0.99, epsilon=1.0, epsilon_decay=0.999, min_epsilon=0.01):
        """SARSA agent for discrete action space trading."""
        self.env = env
        self.alpha = alpha  # Learning rate
        self.gamma = gamma  # Discount factor
        self.epsilon = epsilon  # Exploration rate
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon

        self.q_table = defaultdict(lambda: np.zeros(env.action_space.n))  # Q-table initialization

    def choose_action(self, state):
        """Epsilon-greedy action selection."""
        if np.random.rand() < self.epsilon:
            return self.env.action_space.sample()  # Explore
        return np.argmax(self.q_table[state])  # Exploit

    def train(self, num_episodes=10000):
        """Train the SARSA agent."""
        for episode in range(num_episodes):
            state, _ = self.env.reset()
            state = tuple(state.flatten())  # Convert state to a tuple (hashable)
            action = self.choose_action(state)
            total_reward = 0

            for step in range(1000):
                next_state, reward, done, _, _ = self.env.step(action)
                next_state = tuple(next_state.flatten())

                next_action = self.choose_action(next_state)  # SARSA selects next action

                #SARSA Update Rule
                self.q_table[state][action] += self.alpha * (
                    reward + self.gamma * self.q_table[next_state][next_action] - self.q_table[state][action]
                )

                state, action = next_state, next_action
                total_reward += reward

                if done:
                    break

            #Adjust Epsilon Decay
            self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)

            if episode % 100 == 0:
                print(f"Episode {episode}/{num_episodes}, Total Reward: {total_reward:.2f}")

        print("\nSARSA Training Complete!")

    def evaluate(self, num_episodes=100):
        """Evaluate the trained SARSA agent."""
        total_rewards = []
        action_counts = {"SELL": 0, "HOLD": 0, "BUY": 0}

        for episode in range(num_episodes):
            state, _ = self.env.reset()
            state = tuple(state.flatten())
            total_reward = 0

            for step in range(1000):
                action = np.argmax(self.q_table[state])
                next_state, reward, done, _, _ = self.env.step(action)
                next_state = tuple(next_state.flatten())

                total_reward += reward
                state = next_state

                #Log Actions
                if action == 0:
                    action_counts["SELL"] += 1
                elif action == 1:
                    action_counts["HOLD"] += 1
                else:
                    action_counts["BUY"] += 1

                if done:
                    break

            total_rewards.append(total_reward)
            print(f"Episode {episode+1}/{num_episodes}, Reward: {total_reward:.2f}")

        avg_reward = np.mean(total_rewards)
        print(f"\nAverage Test Reward: {avg_reward:.2f}")
        print("\nAction Distribution:")
        print(f"SELL: {action_counts['SELL']}")
        print(f"HOLD: {action_counts['HOLD']}")
        print(f"BUY: {action_counts['BUY']}")

        #Detect Imbalance in Action Distribution
        if action_counts["SELL"] > action_counts["BUY"] * 5:
            print("Warning: Too much selling compared to buying! Adjust reward incentives.")

In [17]:
#Instantiate SARSA Agent with Adjusted Reward Function
env_discrete_sarsa = DiscreteTradingEnv(df=df, frame_bound=(10, len(df)), window_size=10)
sarsa_agent = SARSAAgent(env_discrete_sarsa)

#Train SARSA Model
print("\nTraining SARSA Model...")
sarsa_agent.train(num_episodes=10000)

#Evaluate SARSA Model
print("\nEvaluating SARSA Model...")
sarsa_agent.evaluate(num_episodes=100)


Training SARSA Model...
Episode 0/10000, Total Reward: 118.77
Episode 9900/10000, Total Reward: 85.23

SARSA Training Complete!

Evaluating SARSA Model...

Average Test Reward: 113.49

Action Distribution:
SELL: 83500
HOLD: 1400
BUY: 15100


In [18]:
print(f"\nInitial Portfolio Value: {env_discrete_sarsa.initial_balance}")
print(f"Final Portfolio Value After Evaluation: {env_discrete_sarsa.portfolio_value}")



Initial Portfolio Value: 100000
Final Portfolio Value After Evaluation: 100083.84625843192


In [19]:
action_counts = {0: 0, 1: 0, 2: 0}  # Sell, Hold, Buy

for state in sarsa_agent.q_table.keys():
    action = np.argmax(sarsa_agent.q_table[state])
    action_counts[action] += 1

print("\nAction Distribution:")
print(f"SELL: {action_counts[0]}")
print(f"HOLD: {action_counts[1]}")
print(f"BUY: {action_counts[2]}")



Action Distribution:
SELL: 7698161
HOLD: 169874
BUY: 343816


In [ ]:
df_volatility = df.copy()
df_volatility['Close'] = df_volatility['Close'] * np.random.uniform(0.9, 1.1, len(df))  # Simulated high volatility

env_stress = DiscreteTradingEnv(df=df_volatility, frame_bound=(10, len(df_volatility)), window_size=10)
sarsa_stress_test = SARSAAgent(env_stress)

print("\nStress Testing SARSA Model...")
sarsa_stress_test.evaluate(num_episodes=100)


In [ ]:
q_values = [np.max(sarsa_agent.q_table[state]) for state in sarsa_agent.q_table.keys()]
plt.plot(q_values)
plt.xlabel("State Index")
plt.ylabel("Max Q-Value")
plt.title("Q-Value Convergence")
plt.show()


In [20]:
print("\nLong-Term Test (1,000 Episodes)...")
sarsa_agent.evaluate(num_episodes=1000)



Long-Term Test (1,000 Episodes)...

Average Test Reward: 84.89

Action Distribution:
SELL: 844000
HOLD: 18000
BUY: 138000


In [20]:
# Download Stock Data
def download_stock_data(ticker, period="720d", interval="1h", max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Attempt {attempt}: Downloading {ticker} stock data...")
            df_live = yf.download(ticker, period=period, interval=interval)
            if not df_live.empty:
                print("Successfully downloaded stock data!")
                df_live.reset_index(inplace=True)
                return df_live
            raise ValueError("Downloaded data is empty. Retrying...")
        except Exception as e:
            print(f"Error: {e}. Retrying in {attempt * 5} seconds...")
            time.sleep(attempt * 5)
    print("Failed to download stock data after multiple attempts.")
    return None

df_live = download_stock_data("TSLA")
if df_live is None:
    print("Using previously saved dataset instead.")
    file_path = '/content/drive/My Drive/teslafeature_engineered_dataset.csv'
    df_live = pd.read_csv(file_path)

# Prepare Dataset
df = df_live.copy()

Successfully downloaded stock data!


In [21]:
# Fix Multi-Index Issues
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Feature Engineering
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['STD_20'] = df['Close'].rolling(window=20).std()
df['Upper_Band'] = df['SMA_20'] + 2 * df['STD_20']
df['Lower_Band'] = df['SMA_20'] - 2 * df['STD_20']
df['Lowest_Low'] = df['Low'].rolling(window=14).min()
df['Highest_High'] = df['High'].rolling(window=14).max()
df['Stoch'] = ((df['Close'] - df['Lowest_Low']) / (df['Highest_High'] - df['Lowest_Low'])) * 100
df.dropna(inplace=True)

# Create Trade Labels
df['Future_Close'] = df['Close'].shift(-10)
df['Price_Change'] = (df['Future_Close'] - df['Close']) / df['Close']
df['Target'] = np.where(df['Price_Change'] > 0.03, 1, 0)
df.dropna(inplace=True)

In [53]:
class ContinuousTradingEnv(gym.Env):
    def __init__(self, df, frame_bound=(10, 100), window_size=10):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.frame_bound = frame_bound
        self.window_size = window_size
        self.current_step = self.frame_bound[0]

        self.action_space = Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        self.observation_space = Box(low=-np.inf, high=np.inf, shape=(13,), dtype=np.float32)

        self.initial_balance = 10_000
        self.portfolio_value = self.initial_balance
        self.shares_held = 0

    def _next_observation(self):
        stock_prices = self.df['Close'].iloc[max(0, self.current_step - self.window_size):self.current_step].values
        if len(stock_prices) < self.window_size:
            stock_prices = np.pad(stock_prices, (self.window_size - len(stock_prices), 0), mode='edge')

        ema_10 = self.df['EMA_10'].iloc[self.current_step] if 'EMA_10' in self.df.columns else 0
        ema_50 = self.df['EMA_50'].iloc[self.current_step] if 'EMA_50' in self.df.columns else 0

        obs = np.concatenate(([self.portfolio_value], stock_prices, [self.shares_held, ema_10, ema_50]), dtype=np.float32)
        obs = obs[:13] if len(obs) > 13 else np.pad(obs, (0, 13 - len(obs)), mode='edge')
        return obs

    def step(self, action):
        """Processes a trading action and updates the portfolio."""
        if self.current_step >= len(self.df) - 1:
            return self._next_observation(), 0, True, False, {}  #Include 'truncated' as False

        self.current_step += 1
        stock_price = self.df['Close'].iloc[self.current_step]

        action = float(action)  #Ensure action is a scalar float

        trade_size = action * 0.1 * self.portfolio_value  #No indexing needed
        shares_traded = abs(trade_size) / max(stock_price, 1e-3)

        reward = 0
        if trade_size > 0 and self.portfolio_value >= shares_traded * stock_price:
            self.shares_held += shares_traded
            self.portfolio_value -= shares_traded * stock_price
            reward += 0.2

        elif trade_size < 0 and self.shares_held > 0:
            shares_sold = min(shares_traded, self.shares_held)
            self.shares_held -= shares_sold
            self.portfolio_value += shares_sold * stock_price
            profit_margin = (stock_price - self.df['Close'].iloc[self.current_step - 1]) / max(self.df['Close'].iloc[self.current_step - 1], 1e-6)
            reward += profit_margin * 100 if profit_margin > 0.005 else -0.05

        return self._next_observation(), reward, False, False, {}  #Ensure correct return structure

    def reset(self, seed=None, options=None):
        """Resets the environment for a new episode."""
        self.current_step = self.frame_bound[0]
        self.portfolio_value = self.initial_balance
        self.shares_held = 0
        return self._next_observation().astype(np.float32), {}


In [54]:
# Initialize TD3 Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env = DummyVecEnv([lambda: ContinuousTradingEnv(df=df, frame_bound=(10, len(df)), window_size=10)])

n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

td3_model = TD3(
    "MlpPolicy",
    env,
    action_noise=action_noise,
    verbose=1,
    learning_rate=0.0001,
    batch_size=128,
    gamma=0.99,
    tau=0.01,
    gradient_steps=2,
    tensorboard_log="./td3_tensorboard/",
    device=device,
)

td3_model.learn(total_timesteps=10000)
td3_model.save("td3_trading_model_v1")


In [55]:
def evaluate_model(model, env, num_steps=100):
    """Evaluates a trained RL model and logs trading signals."""
    result = env.reset()
    if isinstance(result, tuple):  #Handle Gymnasium tuple return
        obs, _ = result
    else:
        obs = result

    trade_log = []  #Store TD3 trading signals
    portfolio_values = []
    total_rewards = np.zeros(env.num_envs)  #VecEnv handles multiple environments

    for step in range(num_steps):
        action, _ = model.predict(obs, deterministic=True)

        #Extract single element safely
        trade_log.append(int(np.squeeze(action)))

        obs, reward, done, _ = env.step(action)  #Ensure action is passed as an array
        total_rewards += reward  #Sum rewards across environments

        #Track portfolio value from the first wrapped environment
        portfolio_value = env.get_attr("portfolio_value")[0]
        portfolio_values.append(portfolio_value)

        if step % 10 == 0:
            print(f"Step {step}: Action: {trade_log[-1]}, Reward: {reward.mean():.4f}, Portfolio: {portfolio_value:.2f}")

        if done.any():  #Handle VecEnv `done`
            result = env.reset()
            obs = result[0] if isinstance(result, tuple) else result

    #Return trade log and portfolio values
    return trade_log, portfolio_values, total_rewards.sum()


In [56]:
#Evaluate TD3 Model
print("\nEvaluating TD3 Model with Portfolio Tracking...")
td3_trade_log, portfolio_values, total_rewards = evaluate_model(td3_model, env, num_steps=100)

#Store TD3 trade results in a separate DataFrame
td3_results_df = pd.DataFrame({
    "TD3_Trade_Signal": td3_trade_log,
    "Portfolio_Value": portfolio_values,
    "Reward": total_rewards
})

#Print trade signal counts
print("\nTD3 Trade Signal Counts:")
print(td3_results_df["TD3_Trade_Signal"].value_counts())

#Display first 20 trades
print("\nTD3 Trading Strategy Completed Successfully!")
print(td3_results_df.head(20))



Evaluating TD3 Model with Portfolio Tracking...
Step 0: Action: 1, Reward: 0.2000, Portfolio: 9000.00
Step 10: Action: 1, Reward: 0.2000, Portfolio: 3138.11
Step 20: Action: 1, Reward: 0.2000, Portfolio: 1094.19
Step 30: Action: 1, Reward: 0.2000, Portfolio: 381.52
Step 40: Action: 1, Reward: 0.2000, Portfolio: 133.03
Step 50: Action: 1, Reward: 0.2000, Portfolio: 46.38
Step 60: Action: 1, Reward: 0.2000, Portfolio: 16.17
Step 70: Action: 1, Reward: 0.2000, Portfolio: 5.64
Step 80: Action: 1, Reward: 0.2000, Portfolio: 1.97
Step 90: Action: 1, Reward: 0.2000, Portfolio: 0.69

TD3 Trade Signal Counts:
TD3_Trade_Signal
1    100
Name: count, dtype: int64

TD3 Trading Strategy Completed Successfully!
    TD3_Trade_Signal  Portfolio_Value  Reward
0                  1      9000.000000    20.0
1                  1      8100.000000    20.0
2                  1      7290.000000    20.0
3                  1      6561.000000    20.0
4                  1      5904.900000    20.0
5                

<ipython-input-53-9fd72917ac17>:36: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  action = float(action)  #Ensure action is a scalar float


In [57]:
if "volatility" not in df.columns:
    print("'volatility' column missing! Computing now...")
    df["volatility"] = df["Close"].pct_change().rolling(20).std()
    df.dropna(inplace=True)

required_features = ['SMA_20', 'STD_20', 'Upper_Band', 'Lower_Band', 'Stoch', 'volatility']
if not all(feature in df.columns for feature in required_features):
    raise ValueError(f"Missing Features: {set(required_features) - set(df.columns)}. Run feature engineering first!")

if 'Target' not in df.columns:
    raise ValueError("'Target' column is missing! Ensure generate_trade_labels(df) was applied.")

print("Features and labels are ready for evaluation.")


Features and labels are ready for evaluation.


In [58]:
#Function to Evaluate Sklearn Models (XGBoost & Random Forest)
def evaluate_sklearn_model(model, df):
    """Evaluates a scikit-learn model (XGBoost or Random Forest) for trading."""
    X = df[feature_columns]
    y = df[target_column]

    predictions = model.predict(X)

    # Simulate Trading Strategy
    portfolio_value = 100000  # Initial balance
    shares_held = 0
    portfolio_values = []

    for i in range(len(df)):
        if predictions[i] == 1 and shares_held == 0:  # BUY signal
            shares_held = portfolio_value / df["Close"].iloc[i]
            portfolio_value = 0
        elif predictions[i] == 0 and shares_held > 0:  # SELL signal
            portfolio_value = shares_held * df["Close"].iloc[i]
            shares_held = 0

        portfolio_values.append(portfolio_value + (shares_held * df["Close"].iloc[i]))

    final_value = portfolio_values[-1]
    profit_loss = final_value - 100000
    return portfolio_values, final_value, profit_loss

In [59]:
#Function to Evaluate SARSA Model
def evaluate_sarsa_model(agent, env, df):
    """Evaluates a trained SARSA agent and calculates profit/loss."""
    result = env.reset()
    obs = result[0] if isinstance(result, tuple) else result  #Fix reset handling

    portfolio_value = 100000  # Initial balance
    shares_held = 0
    portfolio_values = []

    for i in range(len(df)):
        state = tuple(obs.flatten())  # Convert state to tuple
        action = np.argmax(agent.q_table[state]) if state in agent.q_table else 1  # Default to HOLD if state unknown

        if action == 2 and shares_held == 0:  # BUY
            shares_held = portfolio_value / df["Close"].iloc[i]
            portfolio_value = 0
        elif action == 0 and shares_held > 0:  # SELL
            portfolio_value = shares_held * df["Close"].iloc[i]
            shares_held = 0

        portfolio_values.append(portfolio_value + (shares_held * df["Close"].iloc[i]))

        result = env.step(action)
        obs = result[0] if isinstance(result, tuple) else result

    final_value = portfolio_values[-1]
    profit_loss = final_value - 100000
    return portfolio_values, final_value, profit_loss


In [65]:
def evaluate_model(model, env, num_steps=100):
    """Evaluates a trained RL model and ensures correct handling of discrete and continuous actions."""
    result = env.reset()
    obs = np.squeeze(result[0]) if isinstance(result, tuple) else np.squeeze(result)

    portfolio_value = 100000  # Initial balance
    shares_held = 0
    portfolio_values = []

    for step in range(num_steps):
        action, _ = model.predict(obs, deterministic=True)
        action = np.squeeze(action)  #Convert NumPy array to scalar

        #Convert Continuous Action to Discrete if Necessary
        if isinstance(env.action_space, Discrete):  # PPO, A2C, SARSA
            action = int(round(float(action)))  #Ensure action is an integer
            action = np.clip(action, 0, 2)  #Restrict to valid discrete actions (0=SELL, 1=HOLD, 2=BUY)
        else:  # TD3 (Continuous)
            action = float(action)  #Ensure float for continuous environment

        #Apply Trading Logic
        if action == 0 and shares_held > 0:  # SELL
            portfolio_value += shares_held * df["Close"].iloc[step]
            shares_held = 0
        elif action == 2 and shares_held == 0:  # BUY
            shares_held = portfolio_value / df["Close"].iloc[step]
            portfolio_value = 0

        portfolio_values.append(portfolio_value + (shares_held * df["Close"].iloc[step]))

        result = env.step([action])  #Pass as list for VecEnv compatibility
        obs = np.squeeze(result[0]) if isinstance(result, tuple) else np.squeeze(result)

    final_value = portfolio_values[-1]
    profit_loss = final_value - 100000
    return portfolio_values, final_value, profit_loss


In [66]:
#Evaluate Sklearn Models
xgb_results = evaluate_sklearn_model(xgb_model, df)
rf_results = evaluate_sklearn_model(rf_model, df)

#Evaluate RL Models (with Correct Environments)
num_steps = len(df)

td3_results = evaluate_model(td3_model, env, num_steps)  # Continuous Env
ppo_results = evaluate_model(ppo_model, env_discrete, num_steps)  # Discrete Env
a2c_results = evaluate_model(a2c_model, env_discrete, num_steps)  # Discrete Env

#Evaluate SARSA Model
sarsa_results = evaluate_sarsa_model(sarsa_agent, env_discrete_sarsa, df)

#Rank Models Based on Profit/Loss
model_performance = {}

for model_name, result in [
    ("XGBoost", xgb_results),
    ("Random Forest", rf_results),
    ("TD3", td3_results),
    ("PPO", ppo_results),
    ("A2C", a2c_results),
    ("SARSA", sarsa_results),
]:
    if isinstance(result, tuple) and len(result) == 3:
        model_performance[model_name] = result[2]

#Rank Models
sorted_models = sorted(model_performance.items(), key=lambda x: x[1], reverse=True)

#Print Model Rankings
print("\nModel Ranking Based on Profit/Loss:")
for rank, (model, profit) in enumerate(sorted_models, start=1):
    print(f"{rank}. {model}: ${profit:.2f}")



Model Ranking Based on Profit/Loss:
1. XGBoost: $60306.43
2. Random Forest: $56310.20
3. TD3: $0.00
4. A2C: $-2610.65
5. SARSA: $-16046.16
6. PPO: $-23509.08


In [67]:
#Select Winner and Plot Performance
best_model_name = sorted_models[0][0]
best_model_results = locals()[f"{best_model_name.lower()}_results"]

plt.figure(figsize=(12, 6))
plt.plot(best_model_results[0], label=f"{best_model_name} Portfolio Value")
plt.xlabel("Time")
plt.ylabel("Portfolio Value ($)")
plt.title(f"{best_model_name} Trading Model Performance")
plt.legend()
plt.show()

KeyError: 'xgboost_results'